# 키워드·벡터 하이브리드 검색

의미 기반 벡터 검색과 Oracle Text 키워드 검색의 결과를 결합합니다. 벡터 거리만으로 찾은 결과와 비교해, 특정 키워드와 의미 조건이 각각 검색에 어떤 영향을 주는지 살펴봅니다.

## 실습 전 준비

DOC_STORE 원본 문서와 DOC_STORE_CHUNKS, MULTILINGUAL_E5_SMALL, Oracle Text와 하이브리드 벡터 인덱스 기능을 지원하는 RU가 필요합니다. DOC_HB_IDX1은 일반 HNSW 인덱스와 다른 객체입니다.

[공통 환경과 데이터 준비](00.Setup.md)를 먼저 진행합니다. 아래 Python 블록은 같은 세션에서 위에서 아래로 실행하며, 작업 디렉터리는 similarity_search입니다. 코드 블록 자체는 Markdown 화면에서 실행되지 않습니다.

## 처리 흐름

의미 질의 + 키워드 질의 → 하이브리드 인덱스 검색 → UNION으로 후보 결합 → RRF 점수 결합 → 상위 결과

## 하이브리드 인덱스 준비

관리자가 대상 PDB의 Oracle Text 설치·권한·메모리를 준비한 뒤, 실습 계정에서 다음 최소 구성을 실행합니다. 같은 이름의 기존 인덱스가 있으면 용도와 상태부터 확인합니다.

```sql
CREATE HYBRID VECTOR INDEX doc_hb_idx1
ON doc_store(doc)
PARAMETERS('MODEL MULTILINGUAL_E5_SMALL');
```

이는 공식 예제의 최소 구성입니다. 한국어 분석기·청킹 등을 조정하는 확장 구성은 [로컬 자료의 하이브리드 인덱싱 안내](오라클%20벡터%20hybrid%20indexing%20사용%20방법.md)를 참고하되, 설치 RU의 파라미터와 맞춰 확인합니다. 인덱스 생성 후 적재 오류와 동기화 상태를 확인합니다.

DB 연결 후 인덱스 생성 전제 조건을 확인합니다. Hybrid API 지원 여부를 제품명만으로 판단하지 말고 사용 RU와 공식 가이드를 확인합니다.


In [ ]:
import os
from getpass import getpass
# Oracle AI Database 접속 및 라이브러리 로딩
import oracledb
import pandas as pd
from user_functions import display, read_lob
from user_functions import get_conn_cursor, get_db_version, exec_sql, dml_sql, ddl_sql, clean_text1, clean_text2

# DB 접속
user_id = input("DB 사용자: ").strip()
pwd = getpass("DB 비밀번호: ")
db_service = "orclpdb2.subnet0lab23ai.labswoko.oraclevcn.com"
db_host = "localhost"
dbconn = f"{db_host}:1521/{db_service}"

conn, cursor = get_conn_cursor(user_id,pwd,dbconn)

# DB 버전 확인
display(get_db_version(cursor))


## 2.하이브리드 검색

JSON을 SQL 문자열에 직접 이어 붙이지 않습니다. Python의 json.dumps로 검색 명세를 만들고 단일 바인드로 전달합니다. 반환 JSON은 JSON_TABLE로 표 형태로 변환합니다.


In [ ]:
#오라클 벡터 하이브리드 쿼리 샘플
hyb_query = """
SELECT jt.*
FROM JSON_TABLE(
       json_serialize(
         dbms_hybrid_vector.search(
           json(:search_spec)
         )
         RETURNING CLOB
       ),
       '$[*]'
       COLUMNS
         idx          FOR ORDINALITY,
         doc_rowid    VARCHAR2(30)    PATH '$.rowid',
         score        NUMBER          PATH '$.score',
         vector_score NUMBER          PATH '$.vector_score',
         text_score   NUMBER          PATH '$.text_score',
         vector_rank  NUMBER          PATH '$.vector_rank',
         text_rank    NUMBER          PATH '$.text_rank',
         chunk_text   CLOB            PATH '$.chunk_text',
         chunk_id     VARCHAR2(100)   PATH '$.chunk_id',
         paths        CLOB FORMAT JSON PATH '$.paths'
     ) jt
ORDER BY jt.idx
"""
print("하이브리드 쿼리 샘플: :" , hyb_query)


의미 검색 문장과 Oracle Text CONTAINS 검색식을 별도로 입력합니다. UNION은 양쪽 후보를 합치며 RRF는 순위 기반으로 점수를 결합합니다. 키워드만 일치한 행은 chunk_text가 비어 있을 수 있습니다.


In [ ]:
# 조건 입력

semantic_text = input("의미검색 문장 입력 : ")   # 예: 고향사랑 기부 상반기 모금 결과
contains_text = input("키워드 검색 입력 :")  #예 : 고향사랑기부금법(and, or 등 사용 가능)

# 검색 JSON을 직렬화한 뒤 단일 바인드로 전달
import json
search_spec = json.dumps({
    "hybrid_index_name": "DOC_HB_IDX1",
    "vector": {"search_text": semantic_text, "search_mode": "CHUNK"},
    "text": {"contains": contains_text},
    "search_fusion": "UNION",
    "search_scorer": "RRF",
    "return": {"topN": 2, "values": [
        "rowid", "score", "vector_score", "text_score",
        "vector_rank", "text_rank", "chunk_text", "chunk_id", "paths"
    ]}
}, ensure_ascii=False)
cursor.execute(hyb_query, search_spec=search_spec)

rows = []

print("\n=== Hybrid Search Result (Text) ===\n")

for (
    idx, doc_rowid, score, vector_score, text_score,
    vector_rank, text_rank, chunk_text, chunk_id, paths
) in cursor:

    # CLOB → TEXT (1회 read)
    text = read_lob(chunk_text) if chunk_text else ""
    # DataFrame용 row 수집
    rows.append({
        "idx": idx,
        "doc_rowid": doc_rowid,
        "chunk_id": chunk_id,
        "score": score,
        "vector_score": vector_score,
        "text_score": text_score,
        "vector_rank": vector_rank,
        "text_rank": text_rank,
        "chunk_text": text
    })

# -----------------------------
# pandas DataFrame 출력
# -----------------------------
df = pd.DataFrame(rows, columns=["idx", "doc_rowid", "chunk_id", "score", "vector_score", "text_score", "vector_rank", "text_rank", "chunk_text"])

# 컬럼 폭 제한 해제 (문서내용 잘림 방지)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

display(
    df[["doc_rowid", "chunk_id", "score", "chunk_text"]].style.set_properties(
        subset=["chunk_text"],
        **{
            "text-align": "left",
            "white-space": "pre-wrap",
            "word-wrap": "break-word"
        }
    )
)


DOC_STORE_CHUNKS의 벡터 전용 정확 검색 결과와 비교합니다. 하이브리드 인덱스가 자체 생성한 청크와 수동 청크의 경계가 다를 수 있으므로 청크 번호만으로 결과를 비교하지 않습니다.


In [ ]:
# 검색 문장 입력 및 검색
pd.set_option('display.max_colwidth', 200)
print("예: 고향사랑 기부 상반기 모금 결과")
user_sentence = input("찾으실 문장을 입력하세요:")

sim_q = """
Select T.chunked_id, T.chunked_data AS 문서내용
from doc_store_chunks T, doc_store D
where D.docno = T.docno
order by vector_distance(T.embed_vector,
                         dbms_vector.utl_to_embedding(:u_sentence,
                               json('{"provider":"database",
                                      "model":"MULTILINGUAL_E5_SMALL"}')),
                         cosine)
fetch first 2 rows only
"""
sim_result = exec_sql(cursor,sim_q,{"u_sentence":user_sentence})
sim_result = clean_text1(sim_result)
sim_result['문서내용']=sim_result['문서내용'].str.replace('\n', '', regex=False)
display(
    sim_result.head(10).style
    .set_properties(
           **
            {'text-align': 'left',"white-space": "pre-wrap", "word-wrap": "break-word", "max-width": "800px"})
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])
)


## 연결 정리


In [ ]:
cursor.close()
conn.close()


## 결과 확인

- 하이브리드 score는 결합 점수입니다. COSINE 거리나 1-distance와 같은 척도로 해석하지 않습니다.
- rowid로 DOC_STORE의 원본 문서를 확인하고 키워드·의미 각각의 일치 여부를 점검합니다.
- UNION과 INTERSECT의 후보 차이, 키워드 분석기 설정과 한국어 검색 품질을 비교합니다.

## 참고 문서

- [Oracle 하이브리드 검색 전체 예제](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/query-hybrid-vector-indexes-end-end-example.html)
- [Oracle SEARCH 파라미터](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/search.html)

[전체 실습 안내](readme.md) · [공통 함수 설명](user_functions.md)
